Student Name : Eaint Taryar Linlat

# Task 12 — Floorplan GAN: Training a DCGAN on Architectural Images


## Section 0 — Install Dependencies

All core libraries (PyTorch, torchvision, Pillow, matplotlib) are pre-installed
on Colab. We additionally install `torch-fidelity` for a lightweight FID-proxy
metric that quantifies generated image quality beyond visual inspection.

In [1]:
!pip install -q torch torchvision Pillow matplotlib numpy
print('Dependencies ready.')

Dependencies ready.


In [ ]:
import os
import zipfile
import glob
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.utils import make_grid
from PIL import Image

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')

PyTorch  : 2.10.0+cu128
CUDA     : True
GPU      : Tesla T4


## Section 1 — Load and Prepare the Floorplan Dataset

We mount Google Drive, extract the zip, and discover all image files recursively.
Then we visualise a sample of real floorplans to understand the data before training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH    = '/content/floorplans_v2-20251223T170650Z-3-001'
EXTRACT_DIR = './floorplans_data'

if not os.path.exists(EXTRACT_DIR):
    print(f'Extracting {ZIP_PATH} ...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print('Extraction complete.')
else:
    print('Directory already exists, skipping extraction.')

# Find all image files recursively
IMAGE_EXTS = ('*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tif', '*.tiff')
image_paths = []
for ext in IMAGE_EXTS:
    image_paths.extend(glob.glob(os.path.join(EXTRACT_DIR, '**', ext), recursive=True))

print(f'Found {len(image_paths):,} floorplan images.')
if image_paths:
    print(f'Example: {image_paths[0]}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracting /content/floorplans_v2-20251223T170650Z-3-001 ...


FileNotFoundError: [Errno 2] No such file or directory: '/content/floorplans_v2-20251223T170650Z-3-001'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Visualise 8 real floorplans to understand the training data
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('Sample Real Floorplans from Training Set', fontsize=14, fontweight='bold')

sample_indices = np.random.choice(len(image_paths), 8, replace=False)
for ax, idx in zip(axes.flat, sample_indices):
    img = Image.open(image_paths[idx]).convert('RGB')
    ax.imshow(img)
    ax.set_title(f'{img.size[0]}x{img.size[1]}', fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig('real_floorplans_sample.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: real_floorplans_sample.png')

## Section 2 — Hyperparameters and Dataset Class

### Key hyperparameter choices

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `IMAGE_SIZE` | 64 | Good balance of spatial detail vs training time on T4 |
| `LATENT_DIM` | 128 | Larger than baseline (100) → more expressive generator |
| `NGF / NDF` | 64 | Standard DCGAN feature map depth |
| `BATCH_SIZE` | 64 | Standard for DCGAN; fits T4 VRAM with 64×64 images |
| `NUM_EPOCHS` | 100 | Enough for visible floorplan structure; increase for quality |
| `LR` | 0.0002 | From original DCGAN paper |
| `BETA1` | 0.5 | From original DCGAN paper — reduces oscillation |
| `LABEL_SMOOTH` | 0.1 | One-sided label smoothing on real labels → stabilises D |

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
IMAGE_SIZE   = 64     # Resize all floorplans to 64x64
NUM_CHANNELS = 3      # RGB
LATENT_DIM   = 128    # Noise vector dimension (larger than MLP baseline)
NGF          = 64     # Generator feature map base depth
NDF          = 64     # Discriminator feature map base depth

BATCH_SIZE   = 64
NUM_EPOCHS   = 100    # Increase to 200 for higher quality
LR           = 0.0002
BETA1        = 0.5    # Adam beta1 from DCGAN paper
LABEL_SMOOTH = 0.1    # One-sided label smoothing (real labels become 0.9 not 1.0)

DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAMPLE_EVERY = 10     # Save a grid every N epochs

print(f'Device      : {DEVICE}')
print(f'Image size  : {IMAGE_SIZE}x{IMAGE_SIZE} RGB')
print(f'Latent dim  : {LATENT_DIM}')
print(f'Epochs      : {NUM_EPOCHS}')

In [ ]:
class FloorplanDataset(Dataset):
    """
    Custom Dataset for loading floorplan images from a list of file paths.
    Corrupt images are replaced with a black image to avoid crashing the DataLoader.
    """
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        try:
            image = Image.open(self.image_paths[idx]).convert('RGB')
        except Exception:
            image = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), (128, 128, 128))
        if self.transform:
            image = self.transform(image)
        return image


# Transform pipeline:
# 1. Resize to 64x64 (preserves aspect ratio colour then crops)
# 2. Random horizontal flip for data augmentation (floorplans are symmetric)
# 3. Convert to tensor
# 4. Normalise to [-1, 1] to match Generator's Tanh output range
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),   # augmentation — doubles effective dataset
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5],     # mean per channel
                         [0.5, 0.5, 0.5]),    # std per channel → maps [0,1] to [-1,1]
])

dataset    = FloorplanDataset(image_paths, transform=transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=2, pin_memory=True, drop_last=True)

print(f'Dataset size       : {len(dataset):,} images')
print(f'Batches per epoch  : {len(dataloader)}')
print(f'Effective size     : ~{len(dataset)*2:,} (with horizontal flip augmentation)')

## Section 3 — DCGAN Architecture

### Generator
Maps a random latent vector `z` (128-dim) → RGB image (3×64×64) using
**transposed convolutions** that progressively double spatial resolution:
`4×4 → 8×8 → 16×16 → 32×32 → 64×64`

Each upsample block uses: `ConvTranspose2d → BatchNorm → ReLU`

### Discriminator
Maps an RGB image (3×64×64) → probability of being real using
**strided convolutions** that progressively halve spatial resolution:
`64×64 → 32×32 → 16×16 → 8×8 → 1×1`

Key difference from baseline: **Spectral Normalisation** on the Discriminator's
conv layers constrains the Lipschitz constant of D, preventing it from
overwhelming G early in training — the most common cause of GAN collapse.

### Weight Initialisation
From the original DCGAN paper: Conv weights ~ N(0, 0.02); BatchNorm weights ~ N(1, 0.02).

In [ ]:
def weights_init(m):
    """DCGAN weight initialisation (Radford et al. 2015)."""
    classname = m.__class__.__name__
    if 'Conv' in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


class Generator(nn.Module):
    """
    DCGAN Generator: z (LATENT_DIM x 1 x 1) → RGB image (3 x 64 x 64).
    Transposed convolutions upsample: 1x1 -> 4 -> 8 -> 16 -> 32 -> 64.
    """
    def __init__(self, latent_dim=LATENT_DIM, ngf=NGF, nc=NUM_CHANNELS):
        super().__init__()
        self.main = nn.Sequential(
            # z: (latent_dim) x 1 x 1
            nn.ConvTranspose2d(latent_dim, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # → (ngf*8) x 4 x 4

            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # → (ngf*4) x 8 x 8

            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # → (ngf*2) x 16 x 16

            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # → (ngf) x 32 x 32

            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
            # → (nc) x 64 x 64  output in [-1, 1]
        )

    def forward(self, z):
        return self.main(z)


class Discriminator(nn.Module):
    """
    DCGAN Discriminator: RGB image (3 x 64 x 64) → real/fake probability.
    Strided convolutions downsample: 64 -> 32 -> 16 -> 8 -> 4 -> 1.
    SpectralNorm on conv layers prevents D from dominating G (training stability).
    """
    def __init__(self, nc=NUM_CHANNELS, ndf=NDF):
        super().__init__()
        SN = nn.utils.spectral_norm   # spectral normalisation shorthand
        self.main = nn.Sequential(
            # Input: (nc) x 64 x 64
            SN(nn.Conv2d(nc,      ndf,     4, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),
            # → (ndf) x 32 x 32

            SN(nn.Conv2d(ndf,     ndf * 2, 4, 2, 1, bias=False)),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # → (ndf*2) x 16 x 16

            SN(nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False)),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # → (ndf*4) x 8 x 8

            SN(nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False)),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # → (ndf*8) x 4 x 4

            SN(nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False)),
            nn.Sigmoid()
            # → 1 x 1 x 1 (real/fake probability)
        )

    def forward(self, img):
        return self.main(img).view(-1, 1)


# Instantiate and initialise
G = Generator().to(DEVICE)
D = Discriminator().to(DEVICE)
G.apply(weights_init)
D.apply(weights_init)

G_params = sum(p.numel() for p in G.parameters())
D_params = sum(p.numel() for p in D.parameters())
print(f'Generator     : {G_params:,} parameters')
print(f'Discriminator : {D_params:,} parameters')
print(f'Total         : {G_params + D_params:,} parameters')

## Section 4 — Loss Function, Optimisers, and Fixed Noise

**Label smoothing** replaces real labels of `1.0` with `1 - LABEL_SMOOTH = 0.9`.
This prevents the Discriminator from becoming overconfident on real images early
in training, which would produce near-zero gradients for the Generator — the
most common cause of GAN training collapse.

**Fixed noise** (`fixed_z`) is a batch of 16 latent vectors that stays constant
throughout training. By passing the same vectors through G at regular intervals,
we can track *how the same latent points evolve* — making progress visually
interpretable across epochs.

In [ ]:
criterion   = nn.BCELoss()
optimizer_G = optim.Adam(G.parameters(), lr=LR, betas=(BETA1, 0.999))
optimizer_D = optim.Adam(D.parameters(), lr=LR, betas=(BETA1, 0.999))

# Fixed latent vectors for consistent progress tracking across epochs
fixed_z = torch.randn(16, LATENT_DIM, 1, 1, device=DEVICE)

# Label smoothing: real = 0.9, fake = 0.0
REAL_LABEL = 1.0 - LABEL_SMOOTH   # 0.9
FAKE_LABEL = 0.0

print(f'Loss         : BCELoss')
print(f'Optimiser    : Adam (lr={LR}, beta1={BETA1})')
print(f'Real labels  : {REAL_LABEL} (label smoothing = {LABEL_SMOOTH})')
print(f'Fake labels  : {FAKE_LABEL}')
print(f'Fixed noise  : 16 vectors of dim {LATENT_DIM}')

## Section 5 — Training Loop

Each batch follows the standard DCGAN adversarial loop:

```
For each batch:
  ┌── Train Discriminator ──────────────────────────────┐
  │  1. Real images  → D → BCE loss vs label 0.9       │
  │  2. z → G → fake images → D.detach() → BCE vs 0.0  │
  │  3. loss_D = (loss_real + loss_fake) / 2            │
  │  4. Backprop + step optimizer_D                      │
  └─────────────────────────────────────────────────────┘
  ┌── Train Generator ──────────────────────────────────┐
  │  1. z → G → fake images → D → BCE vs label 1.0     │
  │     (trick D into classifying fakes as real)        │
  │  2. Backprop + step optimizer_G                     │
  └─────────────────────────────────────────────────────┘
```

Every `SAMPLE_EVERY` epochs, we pass `fixed_z` through G and save a grid.
This creates a visual history of how image quality improves.

In [ ]:
G_losses = []
D_losses = []
D_real_scores = []   # average D output on real images (should be ~0.5-0.7)
D_fake_scores = []   # average D output on fake images (should be ~0.3-0.5)

os.makedirs('progress_grids', exist_ok=True)

print(f'Training DCGAN for {NUM_EPOCHS} epochs...')
print(f'Dataset: {len(dataset):,} images | Batches/epoch: {len(dataloader)}')
print('-' * 65)

for epoch in range(NUM_EPOCHS):
    epoch_G, epoch_D = 0.0, 0.0
    epoch_Dr, epoch_Df = 0.0, 0.0

    for real_imgs in dataloader:
        real_imgs = real_imgs.to(DEVICE)
        b = real_imgs.size(0)

        real_lbl = torch.full((b, 1), REAL_LABEL, device=DEVICE)
        fake_lbl = torch.full((b, 1), FAKE_LABEL, device=DEVICE)

        # ── Train Discriminator ────────────────────────────────────
        optimizer_D.zero_grad()

        out_real   = D(real_imgs)
        loss_D_real = criterion(out_real, real_lbl)

        z          = torch.randn(b, LATENT_DIM, 1, 1, device=DEVICE)
        fake_imgs  = G(z).detach()   # .detach() stops gradients reaching G
        out_fake   = D(fake_imgs)
        loss_D_fake = criterion(out_fake, fake_lbl)

        loss_D = (loss_D_real + loss_D_fake) / 2
        loss_D.backward()
        optimizer_D.step()

        # ── Train Generator ────────────────────────────────────────
        optimizer_G.zero_grad()

        z         = torch.randn(b, LATENT_DIM, 1, 1, device=DEVICE)
        fake_imgs = G(z)
        out_g     = D(fake_imgs)
        # Use REAL labels (1.0) here — Generator wants D to say 'real'
        loss_G = criterion(out_g, torch.ones(b, 1, device=DEVICE))
        loss_G.backward()
        optimizer_G.step()

        # Accumulate metrics
        epoch_D  += loss_D.item()
        epoch_G  += loss_G.item()
        epoch_Dr += out_real.mean().item()
        epoch_Df += out_fake.mean().item()

    # Epoch averages
    n = len(dataloader)
    G_losses.append(epoch_G / n)
    D_losses.append(epoch_D / n)
    D_real_scores.append(epoch_Dr / n)
    D_fake_scores.append(epoch_Df / n)

    print(f'Ep {epoch+1:3d}/{NUM_EPOCHS} | '
          f'Loss_D: {D_losses[-1]:.4f} | '
          f'Loss_G: {G_losses[-1]:.4f} | '
          f'D(real): {D_real_scores[-1]:.3f} | '
          f'D(fake): {D_fake_scores[-1]:.3f}')

    # ── Save progress grid every SAMPLE_EVERY epochs ──────────────
    if (epoch + 1) % SAMPLE_EVERY == 0 or epoch == 0:
        G.eval()
        with torch.no_grad():
            sample = G(fixed_z).cpu()
        G.train()

        grid = make_grid(sample * 0.5 + 0.5, nrow=4, padding=2)
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(grid.permute(1, 2, 0).numpy().clip(0, 1))
        ax.axis('off')
        ax.set_title(f'Generated Floorplans — Epoch {epoch+1}',
                     fontsize=13, fontweight='bold')
        fname = f'progress_grids/epoch_{epoch+1:03d}.png'
        plt.savefig(fname, dpi=120, bbox_inches='tight')
        plt.show()
        plt.close()

print('\nTraining complete!')

## Section 6 — Training Diagnostics

Two charts diagnose GAN training health:

1. **Loss curves** — Ideally, both losses fluctuate in the range 0.5–1.5.
   If D loss → 0 (D wins), G can't learn. If G loss → 0 (G wins), D provides
   no signal. A healthy GAN oscillates — it rarely converges smoothly.

2. **D score curves** — `D(real)` is D's average output on real images;
   `D(fake)` is D's output on generated images. Healthy equilibrium:
   both should drift toward ~0.5 as training progresses (neither dominates).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
epochs_range = range(1, NUM_EPOCHS + 1)

# Loss curves
axes[0].plot(epochs_range, G_losses, label='Generator',     color='steelblue',  lw=2)
axes[0].plot(epochs_range, D_losses, label='Discriminator', color='firebrick',   lw=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss')
axes[0].set_title('GAN Training Losses', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# D score curves
axes[1].plot(epochs_range, D_real_scores, label='D(real images)',  color='seagreen',   lw=2)
axes[1].plot(epochs_range, D_fake_scores, label='D(fake images)',  color='darkorange',  lw=2)
axes[1].axhline(0.5, color='grey', linestyle='--', lw=1, label='0.5 equilibrium')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Average Discriminator Output')
axes[1].set_title('Discriminator Confidence Over Training', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_diagnostics.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: training_diagnostics.png')

## Section 7 — Generate Final Synthetic Floorplans

With training complete, we sample fresh latent vectors (never seen during training)
and generate 16 synthetic floorplans. These are the model's novel creations.

In [ ]:
NUM_FINAL = 16

G.eval()
with torch.no_grad():
    z_final    = torch.randn(NUM_FINAL, LATENT_DIM, 1, 1, device=DEVICE)
    final_imgs = G(z_final).cpu()

# Display as a grid
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
fig.suptitle('Final Synthetic Floorplans (New Samples, Never Seen in Training)',
             fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    img = final_imgs[i].permute(1, 2, 0).numpy() * 0.5 + 0.5
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(f'Sample {i+1}', fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig('final_synthetic_floorplans.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: final_synthetic_floorplans.png')

## Section 8 — Latent Space Interpolation

One of the key properties of a well-trained GAN is a **smooth latent space**:
interpolating linearly between two latent vectors `z_A` and `z_B` should produce
a gradual, visually coherent transition between two floorplan styles.

Abrupt or noisy transitions indicate the GAN has not learned a continuous
representation — a sign of mode collapse or underfitting.

We generate 8 steps from `z_A` to `z_B` using linear interpolation:
`z_interp = (1 - t) * z_A + t * z_B` for `t` in [0, 1]

In [ ]:
def lerp(z_a, z_b, steps=8):
    """Linear interpolation between two latent vectors."""
    alphas = torch.linspace(0, 1, steps)
    return torch.stack([(1 - a) * z_a + a * z_b for a in alphas])


G.eval()
with torch.no_grad():
    z_a = torch.randn(1, LATENT_DIM, 1, 1, device=DEVICE)
    z_b = torch.randn(1, LATENT_DIM, 1, 1, device=DEVICE)

    # Interpolate: 8 steps from z_a to z_b
    z_interp = lerp(z_a.squeeze(0), z_b.squeeze(0), steps=8).to(DEVICE)
    interp_imgs = G(z_interp).cpu()

fig, axes = plt.subplots(1, 8, figsize=(20, 3))
fig.suptitle('Latent Space Interpolation: z_A → z_B\n'
             '(Smooth transitions = healthy latent space)',
             fontsize=12, fontweight='bold')

for i, ax in enumerate(axes):
    img = interp_imgs[i].permute(1, 2, 0).numpy() * 0.5 + 0.5
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(f't={i/(len(axes)-1):.2f}', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('latent_interpolation.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: latent_interpolation.png')

## Section 9 — Diversity Score (Pixel Variance Heuristic)

A GAN suffering from **mode collapse** generates nearly identical images for all
latent vectors. We detect this with a simple diversity score: the mean per-pixel
standard deviation across a batch of 64 generated images.

- **High diversity score** → generator explores a wide range of outputs ✓
- **Low diversity score** → possible mode collapse ✗



In [ ]:
def diversity_score(imgs_tensor):
    """
    Mean per-pixel standard deviation across a batch.
    Higher = more diverse outputs (less mode collapse).
    imgs_tensor: (N, C, H, W) in [-1, 1]
    """
    # Normalize to [0,1] for interpretability
    imgs = imgs_tensor * 0.5 + 0.5
    # std across the batch dimension (dim=0), then mean across pixels
    return imgs.std(dim=0).mean().item()


G.eval()
with torch.no_grad():
    z_div = torch.randn(64, LATENT_DIM, 1, 1, device=DEVICE)
    fake_div = G(z_div).cpu()

# Real images diversity (sample one batch from dataloader)
real_batch = next(iter(dataloader)).cpu()

gen_diversity  = diversity_score(fake_div)
real_diversity = diversity_score(real_batch)

print('=' * 50)
print('DIVERSITY SCORE (mean per-pixel std across batch)')
print('=' * 50)
print(f'Generated images : {gen_diversity:.4f}')
print(f'Real images      : {real_diversity:.4f}')
print(f'Ratio (gen/real) : {gen_diversity/real_diversity:.3f}')
print()
if gen_diversity / real_diversity > 0.5:
    print('Good diversity — ratio > 0.5 suggests no severe mode collapse.')
else:
    print('Low diversity — possible mode collapse. Consider more training epochs')
    print('or reducing LR and increasing LABEL_SMOOTH.')

## Section 10 — Save Model and Generation Function

We save the full training state (both networks, both optimisers, losses, epoch)
to a single checkpoint file. The `generate_floorplans()` function at the end
can be used in any future session to load the trained Generator and produce
new floorplans without re-training.

In [ ]:
CHECKPOINT_PATH = 'floorplan_dcgan.pth'

torch.save({
    'epoch'                  : NUM_EPOCHS,
    'generator_state_dict'   : G.state_dict(),
    'discriminator_state_dict': D.state_dict(),
    'optimizer_G_state_dict' : optimizer_G.state_dict(),
    'optimizer_D_state_dict' : optimizer_D.state_dict(),
    'G_losses'               : G_losses,
    'D_losses'               : D_losses,
    'hyperparams': {
        'latent_dim' : LATENT_DIM,
        'image_size' : IMAGE_SIZE,
        'ngf'        : NGF,
        'ndf'        : NDF,
    }
}, CHECKPOINT_PATH)

print(f'Checkpoint saved: {CHECKPOINT_PATH}')
print(f'File size: {os.path.getsize(CHECKPOINT_PATH)/1e6:.1f} MB')

In [ ]:
# ── Save individual floorplan images ────────────────────────────────────────
os.makedirs('generated_floorplans', exist_ok=True)

for i in range(NUM_FINAL):
    img = final_imgs[i].permute(1, 2, 0).numpy() * 0.5 + 0.5
    img = np.clip(img, 0, 1)
    Image.fromarray((img * 255).astype(np.uint8)).save(
        f'generated_floorplans/synthetic_{i+1:03d}.png'
    )

print(f'{NUM_FINAL} individual PNG files saved to ./generated_floorplans/')

In [ ]:
def generate_floorplans(checkpoint_path=CHECKPOINT_PATH,
                         num_images=16,
                         save_dir=None):
    """
    Load a trained Generator from a checkpoint and generate new synthetic floorplans.

    Args:
        checkpoint_path : path to the .pth checkpoint file
        num_images      : how many floorplans to generate
        save_dir        : if set, saves individual PNGs to this directory

    Returns:
        imgs_tensor : (num_images, 3, IMAGE_SIZE, IMAGE_SIZE) tensor in [-1, 1]
    """
    ckpt    = torch.load(checkpoint_path, map_location=DEVICE)
    hp      = ckpt.get('hyperparams', {})
    ld      = hp.get('latent_dim', LATENT_DIM)

    gen = Generator(latent_dim=ld).to(DEVICE)
    gen.load_state_dict(ckpt['generator_state_dict'])
    gen.eval()

    with torch.no_grad():
        z    = torch.randn(num_images, ld, 1, 1, device=DEVICE)
        imgs = gen(z).cpu()

    # Display
    cols = min(8, num_images)
    rows = math.ceil(num_images / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 2.5))
    if rows == 1:
        axes = [axes]
    for i, ax in enumerate(np.array(axes).flat):
        if i < num_images:
            img = imgs[i].permute(1, 2, 0).numpy() * 0.5 + 0.5
            ax.imshow(np.clip(img, 0, 1))
        ax.axis('off')
    fig.suptitle(f'{num_images} Newly Generated Synthetic Floorplans',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Optionally save individual files
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        for i in range(num_images):
            img = imgs[i].permute(1, 2, 0).numpy() * 0.5 + 0.5
            Image.fromarray((np.clip(img,0,1)*255).astype(np.uint8)).save(
                os.path.join(save_dir, f'floorplan_{i+1:03d}.png')
            )
        print(f'{num_images} PNGs saved to {save_dir}/')

    return imgs


# ── Demo call ────────────────────────────────────────────────────────────────
generated = generate_floorplans(CHECKPOINT_PATH, num_images=16,
                                 save_dir='new_generation')

---
## Section 11 — Reflection: Insights from Training a Floorplan GAN

### Why DCGAN and Not the MLP Baseline?

The professor's baseline uses a fully-connected GAN on 28×28 MNIST digits.
Floorplans are fundamentally spatial objects: a corridor connects rooms that
are adjacent in pixel space, not in a flattened vector. An MLP GAN treats
every pixel independently — it cannot learn that the top-left corner of an
image is spatially related to the top-right corner. DCGAN's convolutional
architecture shares weights across spatial positions, allowing the model to
learn local structure (wall segments, doorways) that tiles consistently
across the image.

### The Role of Spectral Normalisation

The most significant architectural change over the friend's notebook is
Spectral Normalisation on the Discriminator's convolutional layers. Without
it, the Discriminator frequently achieves near-perfect real/fake discrimination
within the first 5–10 epochs. Once D is that confident, its gradients for G
become vanishingly small — G cannot improve because D is not providing a
useful learning signal. Spectral Norm constrains the Lipschitz constant of
each layer, keeping the Discriminator from becoming too powerful too quickly.

### Reading the Training Diagnostics

A common mistake is interpreting GAN training curves like supervised learning:
lower loss = better. For GANs, the ideal is *equilibrium*, not convergence.
Healthy training shows D and G losses oscillating in a bounded range (roughly
0.5–1.5 for BCE), with `D(real)` and `D(fake)` drifting toward 0.5.
If `D(real) → 1.0` and `D(fake) → 0.0`, the Discriminator has dominated;
if `D(fake) → 1.0`, the Generator has collapsed into a specific output D
can't distinguish from real.

### Latent Space Interpolation as a Quality Test

The interpolation experiment in Section 8 is not merely aesthetic. Smooth
transitions between two latent points indicate that the Generator has learned
a continuous, disentangled representation of floorplan structure. Abrupt jumps
or noisy artefacts in intermediate frames reveal that the latent space has
'holes' — regions where the model has not learned a coherent distribution.

### Limitations and Next Steps

At 64×64 pixels and 100 training epochs, the generated floorplans capture
global structure (rectangular forms, colour distributions) but may lack
fine-grained details (precise wall alignments, furniture placement). Natural
next steps:

- **Higher resolution**: Train at 128×128 or 256×256 with a deeper architecture
- **Progressive growing** (ProGAN): Start at low resolution and progressively
  add layers, which is the current state-of-the-art for high-resolution GANs
- **Conditional GAN**: Condition on building type (residential/commercial)
  or room count to control what kind of floorplan is generated
- **FID score**: A proper Fréchet Inception Distance measurement would provide
  a quantitative quality benchmark comparable across architectures